In [142]:
from pathlib import Path
import sqlite3
import re

import pandas as pd
from tqdm import tqdm

# from pyparsing import (
#     CaselessKeyword,
#     Word,
#     alphas,
#     Opt,
#     nums,
#     Char,
#     Forward,
#     Optional,
#     quoted_string,
#     quotedString,
#     removeQuotes,
#     Group,
#     PrecededBy,
#     ZeroOrMore,
# )

In [ ]:
from pyparsing import (
    CaselessLiteral,
    alphanums,
    infix_notation,
    opAssoc,
    quotedString,
    Word,
    removeQuotes,
)

# from pyparsing import *

and_ = CaselessLiteral("and")
or_ = CaselessLiteral("or")
not_ = CaselessLiteral("not")
search_term = Word(alphanums) | quotedString.set_parse_action(removeQuotes)
search_expression = infix_notation(
    search_term,
    [
        (not_, 1, opAssoc.RIGHT),
        (and_, 2, opAssoc.LEFT),
        (or_, 2, opAssoc.LEFT),
    ],
)

tests = """\
    wood and blue or red
    wood and (blue or red)
    (steel or iron) and "lime green"
    not steel or iron and "lime green"
    not(steel or iron) and "lime green" 
    ("species richness" OR "species number*" OR "number of species" OR "species identity" OR "species composition" OR "species abundance" OR "species divers*") """.splitlines()

for t in tests:
    print(t.strip())
    print(search_expression.parse_string(t)[0])
    print()

wood and blue or red
[['wood', 'and', 'blue'], 'or', 'red']

wood and (blue or red)
['wood', 'and', ['blue', 'or', 'red']]

(steel or iron) and "lime green"
[['steel', 'or', 'iron'], 'and', 'lime green']

not steel or iron and "lime green"
[['not', 'steel'], 'or', ['iron', 'and', 'lime green']]

not(steel or iron) and "lime green"
[['not', ['steel', 'or', 'iron']], 'and', 'lime green']

("species richness" OR "species number*" OR "number of species" OR "species identity" OR "species composition" OR "species abundance" OR "species divers*")
['species richness', 'or', 'species number*', 'or', 'number of species', 'or', 'species identity', 'or', 'species composition', 'or', 'species abundance', 'or', 'species divers*']



In [ ]:
sample = """(A OR B) AND C AND (D OR E)"""
parsed_sample = search_expression.parse_string(sample)[0]
print(sample.strip())
print(search_expression.parse_string(sample)[0])

(A OR B) AND C AND (D OR E)
[['A', 'or', 'B'], 'and', 'C', 'and', ['D', 'or', 'E']]


In [ ]:
for group in parsed_sample:
    if not isinstance(group, str):
        print([x for x in group if x != "or"])
    else:
        print(group)

['A', 'B']
and
C
and
['D', 'E']


In [ ]:
def check_string(text: str):
    print(text.strip())
    print(search_expression.parse_string(text)[0])


def parse_query_string(query_plain_text: str):
    query_parsed = search_expression.parse_string(query_plain_text)[0]
    # for group in query_parsed:
    #     if not isinstance(group, str):
    #         print([x for x in group if x != "or"])
    #     else:
    #         print(group)
    return query_parsed


BIODIVERSITY = """("species richness" OR "species number*" OR "number of species" OR "species identity" OR "species composition" OR "species abundance" OR "species divers*" OR "species evenness" OR diversity OR "biodivers*" OR "biodiversity hotspot*" OR "genetic* divers*" OR "genetic* richness" OR "landscape divers*" OR "landscape structure" OR "landscape composition" OR "habitat divers*" OR "habitat structure" OR "habitat composition" OR "seascape structure" OR "structur* divers*" OR "community composition" OR "ecosystem* divers*" OR "ecosystem* structure" OR "ecosystem composition" OR "phylogenetic* divers*" OR "phylogenetic* richness" OR "number of taxa" OR "taxonomic* divers*" OR "taxonomic* richness" OR "functional* divers*" OR "functional* richness" OR "functional* composition" OR "functional* evenness" OR "functional* dissimilarity" OR "functional* similarity" OR shannon OR simpson OR evenness OR richness OR biomass OR abundance OR density OR "communit*")"""
DEUTSCHLAND = """("German*" OR "central europ*" OR "Lower Saxony" OR Thuringia OR Saarland OR "Berlin*" OR "Brandenburg*" OR "Saxon*" OR "Saxony-Anhalt" OR "Bavar*" OR "Baden-Wuerttemberg" OR Bremen OR Hamburg OR "Hess*" OR "Mecklenburg-Western Pomerania" OR "North RhineWestphalia" OR "Rhineland-Palatinate" OR "Schleswig-Holstein")"""
LEBENSRAUM = """("Forest*" OR "wood*" OR "stock of trees" OR stand OR stands OR "plantation*" OR "maple stand*" OR "birch stand*" OR "beech stand*" OR "rowan stand*" OR "yew stand*" OR "oak stand*" OR "alder stand*" OR "ash stand*" OR "spruce stand*" OR "hornbeam stand*" OR "hazel stand*" OR "chestnut stand*" OR "pine stand*" OR "larch stand*" OR "linden stand*" OR "poplar stand*" OR "sloe stand*" OR "fir stand*" OR "willow stand*" OR "hawthorne stand*" OR "elm stand*" OR deciduous OR coniferous OR "mixed forest*" OR "mixed woodland*" OR coppice OR copse)"""
BASE_KEYWORDS = BIODIVERSITY + " AND " + DEUTSCHLAND + " AND " + LEBENSRAUM
TRENDS = (
    BASE_KEYWORDS
    + " AND "
    + """("trend*" OR "chang*" OR "decreas*" OR "gain*" OR "increas*" OR "los*" OR "loss*" OR "reduc*" OR "declin*" OR "extinct*" OR "develop*" OR "species loss" OR "biodiversity loss" OR "grow*" OR "endangered species" OR "extinct species")"""
)

query_parsed = parse_query_string(TRENDS)
for group in query_parsed:
    if not isinstance(group, str):
        print([x for x in group if x != "or"])
    else:
        print(group)

['species richness', 'species number*', 'number of species', 'species identity', 'species composition', 'species abundance', 'species divers*', 'species evenness', 'diversity', 'biodivers*', 'biodiversity hotspot*', 'genetic* divers*', 'genetic* richness', 'landscape divers*', 'landscape structure', 'landscape composition', 'habitat divers*', 'habitat structure', 'habitat composition', 'seascape structure', 'structur* divers*', 'community composition', 'ecosystem* divers*', 'ecosystem* structure', 'ecosystem composition', 'phylogenetic* divers*', 'phylogenetic* richness', 'number of taxa', 'taxonomic* divers*', 'taxonomic* richness', 'functional* divers*', 'functional* richness', 'functional* composition', 'functional* evenness', 'functional* dissimilarity', 'functional* similarity', 'shannon', 'simpson', 'evenness', 'richness', 'biomass', 'abundance', 'density', 'communit*']
and
['German*', 'central europ*', 'Lower Saxony', 'Thuringia', 'Saarland', 'Berlin*', 'Brandenburg*', 'Saxon*',

In [100]:
df = pd.read_json(
    "/Users/darenasc/projects/dfki/kibad-crawler/data/raw/semantic_scholar/release/2026-08-25/papers/20260828_070435_00070_vxskr_0ad137e4-5397-432c-9b26-5926ecf12afe/json/papers-metadata-0.json"
)
df = df[["paperId", "title", "abstract"]]
text = f"{df.head(1).values[0][1].strip()} {df.head(1).values[0][2].strip()}"

In [145]:
files_path = Path(
    "/Users/darenasc/projects/dfki/kibad-crawler/data/raw/semantic_scholar/release/2026-08-25/papers"
)
json_files = files_path.glob("**/*.json")

In [119]:
# query_parsed[0]
processing_steps = []
for group in query_parsed:
    if not isinstance(group, str):
        # print([x for x in group if x != "or"])
        processing_steps.append([x for x in group if x != "or"])
    else:
        # print(group)
        processing_steps.append(group)

processing_or_steps = [x for x in processing_steps if isinstance(x, list)]

In [120]:
for step in processing_or_steps:
    if isinstance(step, list):
        print(step)

['species richness', 'species number*', 'number of species', 'species identity', 'species composition', 'species abundance', 'species divers*', 'species evenness', 'diversity', 'biodivers*', 'biodiversity hotspot*', 'genetic* divers*', 'genetic* richness', 'landscape divers*', 'landscape structure', 'landscape composition', 'habitat divers*', 'habitat structure', 'habitat composition', 'seascape structure', 'structur* divers*', 'community composition', 'ecosystem* divers*', 'ecosystem* structure', 'ecosystem composition', 'phylogenetic* divers*', 'phylogenetic* richness', 'number of taxa', 'taxonomic* divers*', 'taxonomic* richness', 'functional* divers*', 'functional* richness', 'functional* composition', 'functional* evenness', 'functional* dissimilarity', 'functional* similarity', 'shannon', 'simpson', 'evenness', 'richness', 'biomass', 'abundance', 'density', 'communit*']
['German*', 'central europ*', 'Lower Saxony', 'Thuringia', 'Saarland', 'Berlin*', 'Brandenburg*', 'Saxon*', 'Sa

In [ ]:
check_result = []
for step in processing_or_steps:
    # does not consider `*` operator yet
    check_result.append(any(word.replace("*", "").lower() in text.lower() for word in step))

result = all(check_result)
result

False

In [146]:
for json_file in tqdm(json_files):
    df = pd.read_json(json_file)
    df = df[["paperId", "title", "abstract"]]

    for i, r in tqdm(df.iterrows()):
        text = f"{r['title'].strip()} {r['abstract'].strip() if r['abstract'] else ""}"

        check_result = []
        for step in processing_or_steps:
            check_result.append(
                any(word.replace("*", "").lower() in text.lower() for word in step)
            )

        result = all(check_result)
        if result == True:
            print(r["paperId"])
            # print(f"{i}, {text}")

500it [00:00, 1432.01it/s]
1it [00:00,  2.29it/s]

8cfe395192d4485f68fbfbb3d5511b1beadbea41
811056fa7df79f4d869f5dc4c0904e38531b73e8


500it [00:00, 2694.14it/s]
500it [00:00, 2643.60it/s]
500it [00:00, 2523.59it/s]
500it [00:00, 2296.37it/s]
500it [00:00, 2555.02it/s]
500it [00:00, 2658.19it/s]
7it [00:01,  4.49it/s]

42f621a6a6ed330fd36738ebd0b9456954d2543a
ce5df17ea5b7a0a0c1bec186405a5527abb815a4


500it [00:00, 2587.91it/s]
461it [00:00, 2598.56it/s]
500it [00:00, 2042.94it/s]
10it [00:02,  4.42it/s]

1c699909c19f60b14458cdc73ab12f8cb49a031a


... (30 outputs left out)

1dad947f206b37d96a9616db689e6aacc7347338


500it [00:00, 2739.55it/s]
500it [00:00, 2813.90it/s]
49it [00:10,  5.19it/s]

22a90f47292b7878b9936bc3832603508e1bca94


500it [00:00, 2721.27it/s]
50it [00:10,  4.75it/s]


In [ ]:
print(search_expression.parse_string(filter)[0])

[['species richness', 'or', 'species number*', 'or', 'number of species', 'or', 'species identity', 'or', 'species composition', 'or', 'species abundance', 'or', 'species divers*', 'or', 'species evenness', 'or', 'diversity', 'or', 'biodivers*', 'or', 'biodiversity hotspot*', 'or', 'genetic* divers*', 'or', 'genetic* richness', 'or', 'landscape divers*', 'or', 'landscape structure', 'or', 'landscape composition', 'or', 'habitat divers*', 'or', 'habitat structure', 'or', 'habitat composition', 'or', 'seascape structure', 'or', 'structur* divers*', 'or', 'community composition', 'or', 'ecosystem* divers*', 'or', 'ecosystem* structure', 'or', 'ecosystem composition', 'or', 'phylogenetic* divers*', 'or', 'phylogenetic* richness', 'or', 'number of taxa', 'or', 'taxonomic* divers*', 'or', 'taxonomic* richness', 'or', 'functional* divers*', 'or', 'functional* richness', 'or', 'functional* composition', 'or', 'functional* evenness', 'or', 'functional* dissimilarity', 'or', 'functional* similar

In [ ]:
search_expression = Forward()
search_term = Optional(not_) + (
    Word(alphas)
    | quotedString.setParseAction(removeQuotes)
    | Group(LPAREN + search_expression + RPAREN)
)
searchAnd = Group(search_term + ZeroOrMore(and_ + search_term))
search_expression << Group(searchAnd + ZeroOrMore(or_ + searchAnd))

In [ ]:
import pyparsing as pp

greet = pp.Word(pp.alphas) + "," + pp.Word(pp.alphas) + "!"
for greeting_str in [
    "Hello, World!",
    "Bonjour, Monde!",
    "Hola, Mundo!",
    "Hallo, Welt!",
]:
    greeting = greet.parse_string(greeting_str)
    print(greeting)

['Hello', ',', 'World', '!']
['Bonjour', ',', 'Monde', '!']
['Hola', ',', 'Mundo', '!']
['Hallo', ',', 'Welt', '!']


In [ ]:
AND, OR, NOT = map(CaselessKeyword, "AND OR NOT".split())

# take care not to mistake keywords for identifiers
ident = ~(AND | OR | NOT) + Word(alphas)
boolean_term = Opt(NOT) + ident

# very crude boolean expression - to support parenthesis groups and
# operation hierarchy, use infix_notation
boolean_expr = boolean_term + ((AND | OR) + boolean_term)[...]

# integers that are followed by "." are actually floats
integer = Word(nums) + ~Char(".")

In [ ]:
"""As https://pypi.org/project/sifts/ is leveraging SQLite3 FTS5 capabilities
the crawler can be based on them.

- How to represent queries
- How to load queries
- How to represent '*' and '?' from the words
- re is the logical
"""

# https://pypi.org/project/sifts/
# https://www.sqlite.org/fts5.html
# https://medium.com/@johnidouglasmarangon/full-text-search-in-sqlite-a-practical-guide-80a69c3f42a4

Automating differentially private tabular data synthesis via Bayesian optimization

Generating high-fidelity synthetic tabular data under strict differential privacy (DP) constraints is a critical challenge. The practical deployment of DP-GANs is often derailed by extreme hyperparameter sensitivity, creating a “parameter lottery” where manual or stochastic tuning struggles to balance data utility with rigorous privacy guarantees. To overcome this barrier, we introduce BO-DPCTGAN, an automated framework that formulates hyperparameter tuning as a privacy-aware black-box optimization problem. Leveraging Bayesian optimization with a Gaussian process surrogate, it efficiently navigates the highly non-convex parameter space. Central to our approach is a four-dimensional evaluation system that quantifies statistical fidelity, structural integrity, machine learning utility, and privacy risk. These metrics are dynamically fused via a novel Harmonic mean objective, preventing the optimizer from 

In [8]:
def get_db():
    return sqlite3.connect("papers.db")

In [3]:
conn = get_db()
cursor = conn.cursor()
cursor.execute("CREATE VIRTUAL TABLE papers USING fts5(title, abstract);")
conn.commit()

In [4]:
cursor.execute("SELECT * FROM papers")
cursor.fetchall()

[]

In [ ]:
query_create = """CREATE VIRTUAL TABLE papers USING fts5(title, abstract);"""

In [13]:
query = """("species richness" OR "species number*" OR "number of species" OR "species identity" OR "species composition" OR "species abundance" OR "species divers*" OR "species evenness" OR diversity OR biodivers* OR "biodiversity hotspot*" OR "genetic* divers*" OR "genetic* richness" OR "landscape divers*" OR "landscape structure" OR "landscape composition" OR "habitat divers*" OR "habitat structure" OR "habitat composition" OR "seascape structure" OR "structur* divers*" OR "community composition" OR "ecosystem* divers*" OR "ecosystem* structure" OR "ecosystem composition" OR "phylogenetic* divers*" OR "phylogenetic* richness" OR "number of taxa" OR "taxonomic* divers*" OR "taxonomic* richness" OR "functional* divers*" OR "functional* richness" OR "functional* composition" OR "functional* evenness" OR "functional* dissimilarity" OR "functional* similarity" OR shannon OR simpson OR evenness OR richness OR biomass OR abundance OR density OR communit*)"""
print(query)

("species richness" OR "species number*" OR "number of species" OR "species identity" OR "species composition" OR "species abundance" OR "species divers*" OR "species evenness" OR diversity OR biodivers* OR "biodiversity hotspot*" OR "genetic* divers*" OR "genetic* richness" OR "landscape divers*" OR "landscape structure" OR "landscape composition" OR "habitat divers*" OR "habitat structure" OR "habitat composition" OR "seascape structure" OR "structur* divers*" OR "community composition" OR "ecosystem* divers*" OR "ecosystem* structure" OR "ecosystem composition" OR "phylogenetic* divers*" OR "phylogenetic* richness" OR "number of taxa" OR "taxonomic* divers*" OR "taxonomic* richness" OR "functional* divers*" OR "functional* richness" OR "functional* composition" OR "functional* evenness" OR "functional* dissimilarity" OR "functional* similarity" OR shannon OR simpson OR evenness OR richness OR biomass OR abundance OR density OR communit*)


In [25]:
"""
- capture strings
- identify startswith, endswith or normal
- list of words
"""

# print("AND" in query)
# print(query.replace('"', "").replace("(", "").replace(")", "").split(" OR "))
words = query.replace('"', "").replace("(", "").replace(")", "").split(" OR ")
for word in words:
    if "*" in word:
        print(f"{word} | {word.count('*')} * | {word.count(' ')} words")
    else:
        print(word)

species richness
species number* | 1 * | 1 words
number of species
species identity
species composition
species abundance
species divers* | 1 * | 1 words
species evenness
diversity
biodivers* | 1 * | 0 words
biodiversity hotspot* | 1 * | 1 words
genetic* divers* | 2 * | 1 words
genetic* richness | 1 * | 1 words
landscape divers* | 1 * | 1 words
landscape structure
landscape composition
habitat divers* | 1 * | 1 words
habitat structure
habitat composition
seascape structure
structur* divers* | 2 * | 1 words
community composition
ecosystem* divers* | 2 * | 1 words
ecosystem* structure | 1 * | 1 words
ecosystem composition
phylogenetic* divers* | 2 * | 1 words
phylogenetic* richness | 1 * | 1 words
number of taxa
taxonomic* divers* | 2 * | 1 words
taxonomic* richness | 1 * | 1 words
functional* divers* | 2 * | 1 words
functional* richness | 1 * | 1 words
functional* composition | 1 * | 1 words
functional* evenness | 1 * | 1 words
functional* dissimilarity | 1 * | 1 words
functional* simil